In [1]:
import json
import numpy as np
import pandas as pd

In [2]:
SURROGATE_CSV = "data/adult_ready_for_surrogates.csv"
ACFX_TABLE_CSV = "data/acfx_table.csv"        # single table for ACFX (X + y)
ACFX_MAPS_JSON = "data/acfx_maps.json"        # for decoding later

# Optional: ACFX export to validate
ACFX_EXPORT_CSV = "data/acfx_results.csv"



In [3]:
df = pd.read_csv(SURROGATE_CSV, low_memory=False)

# continuous target used in surrogate training
y_sur = df["value"].astype(float)

# normalize params_params_* -> params_*
df = df.rename(columns=lambda c: c.replace("params_params_", "params_"))

FEATURES = [
    "params_alpha","params_booster","params_colsample_bytree","params_gamma",
    "params_grow_policy","params_lambda","params_learning_rate","params_max_delta_step",
    "params_max_depth","params_min_child_weight","params_n_estimators","params_normalize_type",
    "params_scale_pos_weight","params_rate_drop","params_sample_type","params_skip_drop",
    "params_subsample","user_attrs_decision_threshold","user_attrs_feature_fraction",
    "user_attrs_resampling_strategy","user_attrs_sample_rows", "params_selftrain_max_iter",
    "params_selftrain_threshold", "user_attrs_labeled_ratio","user_attrs_missing_rate",
    "soft_risk_preference", "soft_decision_speed"
]

X = df[FEATURES].copy()


In [4]:
# DART conditionals
mask_not_dart = X["params_booster"].astype(str) != "dart"
for c in ["params_rate_drop", "params_skip_drop"]:
    X.loc[mask_not_dart, c] = 0.0
for c in ["params_sample_type", "params_normalize_type"]:
    X.loc[mask_not_dart, c] = "NA"

# Defaults (same as surrogate prep)
X["params_scale_pos_weight"] = pd.to_numeric(X["params_scale_pos_weight"], errors="coerce").fillna(1.0)
X["user_attrs_decision_threshold"] = pd.to_numeric(X["user_attrs_decision_threshold"], errors="coerce").fillna(0.5)


In [5]:
bins   = [0, 0.5, 0.6, 0.7, 1.0]
labels = [0, 1, 2, 3]

cat_value = pd.cut(y_sur.clip(0, 1), bins=bins, labels=labels, include_lowest=True).astype(int)
assert not cat_value.isna().any()


In [6]:
CAT_COLS = [
    "params_booster","params_grow_policy","params_sample_type","params_normalize_type",
    "user_attrs_resampling_strategy","soft_risk_preference","soft_decision_speed",
]

X_num = X.copy()
maps = {"categorical": {}, "numeric_fill": {}, "target": {"name": "cat_value", "bins": bins, "labels": labels}}

# Encode categorical columns -> int codes
for c in CAT_COLS:
    s = X_num[c].fillna("NA").astype(str)
    cats = sorted(s.unique().tolist())
    code = {v:i for i,v in enumerate(cats)}
    X_num[c] = s.map(code).astype(int)
    maps["categorical"][c] = cats

# Fill numeric NaNs -> median (deterministic)
for c in X_num.columns.difference(CAT_COLS):
    col = pd.to_numeric(X_num[c], errors="coerce")
    fill = float(col.median()) if col.isna().any() else 0.0
    if np.isnan(fill): fill = 0.0
    X_num[c] = col.fillna(fill).astype(float)
    maps["numeric_fill"][c] = fill

# Build the single ACFX table (features + target)
acfx_table = X_num.copy()
acfx_table["cat_value"] = cat_value.values

# Final sanity: fully numeric, no NaNs
assert not acfx_table.isna().any().any()
assert all(pd.api.types.is_numeric_dtype(acfx_table[c]) for c in acfx_table.columns)

acfx_table.to_csv(ACFX_TABLE_CSV, index=False)
with open(ACFX_MAPS_JSON, "w", encoding="utf-8") as f:
    json.dump(maps, f, indent=2)

print("Saved:", ACFX_TABLE_CSV, "shape=", acfx_table.shape)
print("Saved:", ACFX_MAPS_JSON)
display(acfx_table.head(3).T)


Saved: data/acfx_table.csv shape= (11077, 28)
Saved: data/acfx_maps.json


,0,1,2
params_alpha,60.530821,0.507941,60.530821
params_booster,0.000000,1.000000,0.000000
params_colsample_bytree,0.833380,0.693547,0.833380
params_gamma,4.375872,5.684339,4.375872
params_grow_policy,1.000000,0.000000,1.000000
params_lambda,22.420124,0.000130,22.420124
params_learning_rate,0.003691,0.073092,0.003691
params_max_delta_step,10.000000,3.000000,10.000000
params_max_depth,8.000000,5.000000,8.000000
params_min_child_weight,8.663280,2.335883,8.663280
